# AG_PRAXIS NB07 — Class Balancing

Nineteen classes, and the largest holds more than two thousand times the rows of the
smallest. A model trained on a corpus shaped like that can be right about almost every
row and still miss the rare classes entirely, because the rare classes contribute almost
nothing to the number it is minimising. The scores in this project are reported two ways
for that reason: weighted, which is what the common classes did, and macro, which counts
a class with a few dozen windows the same as a class with sixty thousand.

This notebook tries five ways of changing that, one per run, and reports what each one
costs as well as what it gains.

Three of them change what the model learns. Weighting the classes in the loss makes a
rare window count for more. Focal loss makes any window the model already answers
confidently count for less. Oversampling shows it a rare window several times an epoch
instead of once. Two of them change nothing about training and only change how a trained
model's output is turned into an answer: subtracting the class priors from the scores,
and dividing each class's score by a threshold chosen for it.

Every run is one change from the same parent. Same windows, same split, same
architecture, same forty-four columns, same ten epochs at batch 32, same seed. Five runs,
five sets of per-class scores.

Nothing here is a test. No run is declared better than another, no significance is
computed and no threshold decides a pass or a fail. What this notebook produces is a
table of what each intervention did to each class, with the classes that gain and the
classes that lose in the same view, because an intervention that lifts four rare classes
while dropping a common one has done both of those things and reporting only the first
would be choosing what to look at.

The data sits on Drive and the code sits in the repository, so the first block mounts one
and clones the other, and records the commit it is running from.

In [1]:
import os
import subprocess
import sys
from datetime import date
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AGREWAL14/AG_PRAXIS.git"
NOTEBOOK = "AG_PRAXIS_NB07_class_balancing.ipynb"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/repo")
    if REPO_ROOT.exists():
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / "config" / "base.yaml").exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def git(*args):
    return subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args], capture_output=True, text=True
    ).stdout.strip()


GIT_SHA = git("rev-parse", "--short", "HEAD")
GIT_BRANCH = git("rev-parse", "--abbrev-ref", "HEAD")
GIT_DIRTY = bool(git("status", "--porcelain"))
RUN_DATE = date.today().isoformat()

print(f"colab     : {IN_COLAB}")
print(f"repo root : {REPO_ROOT}")
print(f"git sha   : {GIT_SHA} on {GIT_BRANCH}" + ("   WORKING TREE DIRTY" if GIT_DIRTY else ""))
print(f"run date  : {RUN_DATE}")

Mounted at /content/drive
colab     : True
repo root : /content/repo
git sha   : 45f376d on main
run date  : 2026-08-08


The parameters come from `config/base.yaml` and the shape of the data comes from the
manifest the preprocessing step wrote. Neither is retyped.

Two more files are read, and both belong to the run every run here is one change from.
Its configuration is read so that these five inherit from it rather than restate it, and
its metrics are read so that every comparison below quotes numbers off disk. That run is
the parent in the strict sense the project uses: each configuration here is its
configuration with exactly one key different.

In [2]:
import gc
import json
import random
import textwrap
import time

import numpy as np
import pandas as pd

from baselines import mohammadi as mo
from src import interventions as iv
from src import inventory as inv
from src import runs as rn
from src import sequence as sq

CFG = inv.load_config(REPO_ROOT)

SEED = CFG["seed"]
BATCH_SIZE = int(CFG["training"]["batch_size"])
EPOCHS = int(CFG["training"]["epochs"])
WINDOW = int(CFG["sequence"]["window"])
STRIDE = int(CFG["sequence"]["stride"])
ARTIFACTS = Path(CFG["paths"]["artifacts"])
OUT_DIRS = {"fast": ARTIFACTS / "NB07_fast", "full": ARTIFACTS / "NB07"}
NB04_DIRS = {"fast": ARTIFACTS / "NB04_fast", "full": ARTIFACTS / "NB04"}

if IN_COLAB and not ARTIFACTS.exists():
    raise FileNotFoundError(
        f"{ARTIFACTS} does not exist. Drive is not mounted, or the artefacts path in "
        "config/base.yaml is wrong. Nothing this notebook writes would survive."
    )


def first_existing(candidates, what):
    found = next((Path(p) for p in candidates if Path(p).exists()), None)
    if found is None:
        raise FileNotFoundError(f"{what} not found. Looked in: {[str(p) for p in candidates]}")
    return found


MANIFEST_PATH = first_existing(
    [
        REPO_ROOT / "data" / "processed" / "NB04_manifest.json",
        NB04_DIRS["full"] / "NB04_manifest.json",
    ],
    "NB04_manifest.json",
)
PARENT_DIR = first_existing(
    [REPO_ROOT / "data" / "processed" / "NB06", ARTIFACTS / "NB06" / "sequence_cnn_lstm_19class"],
    "the parent run's config.json and metrics.json",
)

MANIFEST = json.loads(MANIFEST_PATH.read_text())
PARENT = json.loads((PARENT_DIR / "config.json").read_text())
PARENT_METRICS = json.loads((PARENT_DIR / "metrics.json").read_text())

FEATURES = list(MANIFEST["columns"]["kept"])
CLASSES = sorted(MANIFEST["arrays"]["sequences_train"]["by_class"])
SEQUENCES = {
    partition: dict(MANIFEST["arrays"][f"sequences_{partition}"]["by_class"])
    for partition in ("train", "val", "test")
}

pd.set_option("display.max_rows", 400)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 90)

print(f"seed           : {SEED}")
print(f"window, stride : {WINDOW} records, {STRIDE} records")
print(f"batch, epochs  : {BATCH_SIZE}, {EPOCHS}")
print(f"manifest       : {MANIFEST_PATH}")
print(f"parent         : {PARENT['run_id']} from {PARENT_DIR}")
print(f"  its scores   : accuracy {PARENT_METRICS['accuracy']:.4f}, weighted F1 "
      f"{PARENT_METRICS['weighted_f1']:.4f}, macro F1 {PARENT_METRICS['macro_f1']:.4f}")
print(f"  it trained on: {PARENT_METRICS['n_train']:,} windows, scored on "
      f"{PARENT_METRICS['n_test']:,}")
print(f"features       : {len(FEATURES)}, classes {len(CLASSES)}")
print(f"fast pass to   : {OUT_DIRS['fast']}")
print(f"full pass to   : {OUT_DIRS['full']}")

assert len(FEATURES) == 44 and len(CLASSES) == 19
assert PARENT_METRICS["labels"] == CLASSES, (
    "the parent was scored on a different list of classes, so its per-class figures cannot "
    "be lined up with these runs"
)
assert PARENT["split"] == "two_tier" and PARENT["seed"] == SEED
assert PARENT["batch_size"] == BATCH_SIZE and PARENT["epochs"] == EPOCHS
assert MANIFEST["sequences"]["window"] == WINDOW and MANIFEST["sequences"]["stride"] == STRIDE

seed           : 42
window, stride : 50 records, 25 records
batch, epochs  : 32, 10
manifest       : /content/repo/data/processed/NB04_manifest.json
parent         : sequence_cnn_lstm_19class from /content/repo/data/processed/NB06
  its scores   : accuracy 0.8197, weighted F1 0.8064, macro F1 0.7138
  it trained on: 249,061 windows, scored on 49,159
features       : 44, classes 19
fast pass to   : /content/drive/MyDrive/AG_PRAXIS_artifacts/NB07_fast
full pass to   : /content/drive/MyDrive/AG_PRAXIS_artifacts/NB07


Two sets of classes get their own reporting, and both are fixed before anything runs.

The first is the five classes a single-record convolutional model fails to detect, which
is the set the project's second hypothesis is about, at a threshold of F1 0.50. Every run
here reports its per-class F1 on those five against the parent's.

The second is the eight flooding classes, the ones with the most rows. They get their own
block because the parent's own result showed that a change made to help the rare classes
can move these instead, and because the hypothesis those five classes belong to says
nothing about what happens outside them. Reporting the eight in the same file as the five
means the gain and the cost of an intervention can be read without opening a second
document. Each run reports the per-class F1 of all eight and the mean across them.

The third group is the classes too thin to read. A class with a few dozen windows in the
partition it is scored on produces an F1 that can only take a few values, so those
classes are flagged in every run's metrics and their figures are reported without being
treated as measurements.

In [3]:
DETECTED_AT = 0.50
THIN_BELOW = 50

H2_CLASSES = [
    "Recon-VulScan",
    "Recon-OS_Scan",
    "MQTT-DDoS-Publish_Flood",
    "Spoofing",
    "MQTT-Malformed_Data",
]
VOLUMETRIC = [
    "DDoS-ICMP", "DDoS-SYN", "DDoS-TCP", "DDoS-UDP",
    "DoS-ICMP", "DoS-SYN", "DoS-TCP", "DoS-UDP",
]
FLAGGED = {
    "Recon-Ping_Sweep": (
        "Too few sequences in the partitions it is scored on for an F1 to take more than a "
        "few values. Reported and not interpreted."
    ),
    "Recon-VulScan": (
        "Thin in every partition. Kept in the five compared classes, and its per-class figure "
        "carries this caveat wherever it is quoted."
    ),
    "MQTT-Malformed_Data": (
        "Thin in every partition. Kept in the five compared classes, and its per-class figure "
        "carries this caveat wherever it is quoted."
    ),
}

start = pd.DataFrame(
    {
        "class": H2_CLASSES + VOLUMETRIC,
        "group": ["the five"] * len(H2_CLASSES) + ["volumetric"] * len(VOLUMETRIC),
        "parent_f1": [PARENT_METRICS["per_class_f1"][c] for c in H2_CLASSES + VOLUMETRIC],
        "train_windows": [SEQUENCES["train"][c] for c in H2_CLASSES + VOLUMETRIC],
        "test_windows": [SEQUENCES["test"][c] for c in H2_CLASSES + VOLUMETRIC],
        "thin": [c in FLAGGED for c in H2_CLASSES + VOLUMETRIC],
    }
)
print("where the parent stands on both sets, and what each run is measured against")
print(start.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()
parent_five = np.mean([PARENT_METRICS["per_class_f1"][c] for c in H2_CLASSES])
parent_eight = np.mean([PARENT_METRICS["per_class_f1"][c] for c in VOLUMETRIC])
print(f"parent mean F1 across the five      : {parent_five:.4f}")
print(f"parent mean F1 across the eight     : {parent_eight:.4f}")
print(f"parent macro F1 across all nineteen : {PARENT_METRICS['macro_f1']:.4f}")
print(f"of the five, detected at {DETECTED_AT:.2f}      : "
      f"{sum(PARENT_METRICS['per_class_f1'][c] >= DETECTED_AT for c in H2_CLASSES)} of "
      f"{len(H2_CLASSES)}")
print(f"of the eight, detected at {DETECTED_AT:.2f}     : "
      f"{sum(PARENT_METRICS['per_class_f1'][c] >= DETECTED_AT for c in VOLUMETRIC)} of "
      f"{len(VOLUMETRIC)}")

assert not set(H2_CLASSES) & set(VOLUMETRIC), "a class is in both groups"
assert set(H2_CLASSES) <= set(CLASSES) and set(VOLUMETRIC) <= set(CLASSES)
assert len(VOLUMETRIC) == 8 and len(H2_CLASSES) == 5

where the parent stands on both sets, and what each run is measured against
                  class      group  parent_f1  train_windows  test_windows  thin
          Recon-VulScan   the five     0.5385             86            18  True
          Recon-OS_Scan   the five     0.6061            577           123 False
MQTT-DDoS-Publish_Flood   the five     0.0796           1008           215 False
               Spoofing   the five     0.7653            497           104 False
    MQTT-Malformed_Data   the five     0.8421            191            40  True
              DDoS-ICMP volumetric     0.7721          61487          7826 False
               DDoS-SYN volumetric     0.8926          24198          6894 False
               DDoS-TCP volumetric     0.8137          24135          7302 False
               DDoS-UDP volumetric     0.9881          65426          6255 False
               DoS-ICMP volumetric     0.3178          12403          3936 False
                DoS-SYN volumetri

The seed is set here, and again immediately before each model is built, inside the
statement that fits it. Five runs start from the same weights and see the same shuffle,
so a difference between two of them is the intervention rather than where they started.

In [4]:
import keras
import tensorflow as tf

random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

GPUS = tf.config.list_physical_devices("GPU")

print(f"seeded with : {SEED}")
print(f"tensorflow  : {tf.__version__}")
print(f"keras       : {keras.__version__}")
print(f"gpu         : {[d.name for d in GPUS] if GPUS else 'none, this will be slow'}")

seeded with : 42
tensorflow  : 2.20.0
keras       : 3.13.2
gpu         : ['/physical_device:GPU:0']


The model is the parent's, unchanged, in all five runs.

A convolutional encoder reads each of the fifty records in a window on its own and turns
it into 128 numbers, one LSTM reads those fifty results in order, and a dense softmax
gives nineteen scores. The encoder is the published single-record network with its
classification layer removed, which is checked below rather than assumed: the cell
rebuilds that network from scratch and compares the two layer by layer.

None of the five interventions touches the architecture. Three of them change the loss or
the rows it is computed over, and two of them change what happens to the nineteen scores
after training is finished.

In [5]:
SPECIMEN = sq.build_model(len(FEATURES), len(CLASSES), window=WINDOW, lstm_units=sq.LSTM_UNITS)
SPECIMEN.summary()

ENCODER_CHECK = sq.encoder_matches_baseline(SPECIMEN, len(FEATURES), len(CLASSES))
print()
print("the record encoder against the published network, layer by layer")
print(pd.DataFrame(ENCODER_CHECK["rows"]).to_string(index=False))
print()
print(f"parameters: {SPECIMEN.count_params():,}, the same in every run below")
print(f"input {SPECIMEN.input_shape}, output {SPECIMEN.output_shape}")

assert ENCODER_CHECK["agrees"]
assert SPECIMEN.input_shape == (None, WINDOW, len(FEATURES), 1)
assert int(SPECIMEN.count_params()) == int(PARENT_METRICS["n_parameters"]), (
    "this model does not have the same number of parameters as the parent, so it is not "
    "the same architecture"
)

del SPECIMEN
gc.collect()

Model: "mohammadi_cnn_lstm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ per_record (TimeDistributed)    │ (None, 50, 128)        │        80,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ across_the_window (LSTM)        │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classifier (Dense)              │ (None, 19)             │         2,451 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 214,227 (836.82 KB)

 Trainable params: 214,227 (836.82 KB)

 Non-trainable params: 0 (0.00 B)


the record encoder against the published network, layer by layer
 position     baseline      encoder baseline_output encoder_output  baseline_params  encoder_params
        0       Conv1D       Conv1D  (None, 42, 32) (None, 42, 32)              128             128
        1 MaxPooling1D MaxPooling1D  (None, 21, 32) (None, 21, 32)                0               0
        2       Conv1D       Conv1D  (None, 19, 64) (None, 19, 64)             6208            6208
        3 MaxPooling1D MaxPooling1D   (None, 9, 64)  (None, 9, 64)                0               0
        4      Flatten      Flatten     (None, 576)    (None, 576)                0               0
        5        Dense        Dense     (None, 128)    (None, 128)            73856           73856

parameters: 214,227, the same in every run below
input (None, 50, 44, 1), output (None, 19)


3791

The five interventions, and where each one enters.

**Class-weighted loss.** Each class gets a weight of `N / (K * n)`, so a class with a
thousandth of the rows counts a thousand times as much per row and the average row counts
exactly what it did before. Normalising is what keeps this a single change: weights that
average to something other than one would scale every gradient in the run.

**Focal loss.** Cross-entropy asks for the same attention on every window. Focal loss
multiplies each window's term by `(1 - p)` raised to a power, where `p` is the probability
given to the right answer, so a window the model already gets right contributes almost
nothing. Most of the windows it already gets right belong to the classes with the most
rows.

**Logit adjustment.** Nothing about training changes. After training, each class's score
has the log of that class's share of the training rows subtracted from it, so a class is
chosen when its score is high relative to how often it appears rather than high outright.
The shares are counted on the training rows only.

**Window-level resampling.** Every class is drawn up to the median class size, with
replacement, so a rare window is seen several times an epoch. The median rather than the
largest class: drawing everything up to the largest would multiply the training set five
times over and would repeat the smallest class's two dozen windows tens of thousands of
times, which would say more about those two dozen windows than about the class.

**Per-class thresholds.** Nothing about training changes. Each class's score is divided by
a number chosen for that class, and the largest of the divided scores wins. A threshold of
one leaves a class exactly where it was, so a run with every threshold at one decides the
way the ordinary rule does. The numbers are chosen on the validation partition, one pass
over the classes from rarest to commonest, keeping whichever grid value gives the best
macro-F1. The test partition is not read while they are being chosen and is scored once,
afterwards.

In [6]:
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.25
LOGIT_TAU = 1.0
THRESHOLD_GRID = [round(float(v), 2) for v in np.arange(0.05, 1.0, 0.05)]

INTERVENTIONS = {
    "class_weighted_loss": {
        "key": "class_weight",
        "value": "inverse frequency, N / (K * n), normalised to mean 1",
        "changes": "what the model learns",
        "needs_validation": False,
    },
    "focal_loss": {
        "key": "loss",
        "value": f"categorical_focal_crossentropy, gamma {FOCAL_GAMMA}, alpha {FOCAL_ALPHA}",
        "changes": "what the model learns",
        "needs_validation": False,
    },
    "logit_adjustment": {
        "key": "decision_rule",
        "value": f"logit adjustment, tau {LOGIT_TAU}, priors counted on the training windows",
        "changes": "how a trained model decides",
        "needs_validation": False,
    },
    "threshold_tuning": {
        "key": "decision_rule",
        "value": "one threshold per class, tuned on validation, largest divided score wins",
        "changes": "how a trained model decides",
        "needs_validation": True,
    },
    "window_resampling": {
        "key": "resampling",
        "value": "window-level oversampling to the median class size, with replacement",
        "changes": "what the model learns",
        "needs_validation": False,
    },
}


def prepare(run_id, data):
    """The hooks and the training windows this intervention needs, and what it did."""
    hooks = {
        "X_train": data["X_train"],
        "y_train": data["y_train"],
        "loss": None,
        "class_weight": None,
        "decision_rule": None,
    }
    observed = {}

    if run_id == "class_weighted_loss":
        weights = iv.inverse_frequency_weights(hooks["y_train"], len(CLASSES))
        hooks["class_weight"] = weights
        observed["class_weights"] = {
            CLASSES[code]: round(weight, 4) for code, weight in weights.items()
        }

    elif run_id == "focal_loss":
        hooks["loss"] = iv.focal_loss(FOCAL_GAMMA, FOCAL_ALPHA)
        observed["focal"] = {"gamma": FOCAL_GAMMA, "alpha": FOCAL_ALPHA}

    elif run_id == "logit_adjustment":
        prior = iv.priors(hooks["y_train"], len(CLASSES))
        record = {
            "rule": "log score minus tau times log prior, largest wins",
            "tau": LOGIT_TAU,
            "priors_counted_on": "the training windows of this run",
            "priors": {CLASSES[code]: round(float(p), 8) for code, p in enumerate(prior)},
        }
        hooks["decision_rule"] = lambda model: (
            iv.logit_adjusted_decision(prior, LOGIT_TAU),
            record,
        )
        observed["priors_counted_on"] = record["priors_counted_on"]

    elif run_id == "threshold_tuning":
        validation = data["validation"]

        def rule(model):
            probabilities = model.predict(
                validation["X"], batch_size=PREDICT_BATCH, verbose=0
            )
            thresholds, record = iv.tune_thresholds(
                probabilities,
                validation["y"],
                n_classes=len(CLASSES),
                grid=THRESHOLD_GRID,
                labels=CLASSES,
            )
            return iv.threshold_decision(thresholds), record

        hooks["decision_rule"] = rule
        observed["tuned_on"] = f"{len(validation['y']):,} validation windows"

    elif run_id == "window_resampling":
        index, target = iv.oversample_index(hooks["y_train"], len(CLASSES), seed=SEED)
        before = hooks["y_train"]
        hooks["X_train"] = data["X_train"][index]
        hooks["y_train"] = data["y_train"][index]
        observed["resampling"] = iv.resampling_record(
            before, hooks["y_train"], len(CLASSES), target=target, classes=CLASSES
        )
        gc.collect()

    else:
        raise KeyError(f"{run_id} is not one of the five interventions")

    return hooks, observed


print(pd.DataFrame(
    [
        {
            "run_id": run_id,
            "the one key": spec["key"],
            "changes": spec["changes"],
            "reads validation": spec["needs_validation"],
        }
        for run_id, spec in INTERVENTIONS.items()
    ]
).to_string(index=False))
print()
print(f"threshold grid: {THRESHOLD_GRID[0]} to {THRESHOLD_GRID[-1]} in steps of 0.05, "
      f"{len(THRESHOLD_GRID)} values")
print("Only the threshold run reads the validation partition, and it reads it to choose")
print("thresholds, never to score.")

             run_id   the one key                     changes  reads validation
class_weighted_loss  class_weight       what the model learns             False
         focal_loss          loss       what the model learns             False
   logit_adjustment decision_rule how a trained model decides             False
   threshold_tuning decision_rule how a trained model decides              True
  window_resampling    resampling       what the model learns             False

threshold grid: 0.05 to 0.95 in steps of 0.05, 19 values
Only the threshold run reads the validation partition, and it reads it to choose
thresholds, never to score.


A run is a configuration, and the configuration is the ledger entry.

The parent is read from the file the parent run wrote and is used exactly as it is, not
restated. Each of the five copies it and sets one key: the class weighting, the loss, the
decision rule, or the resampling. `assert_single_change` compares the two configurations
in both directions and raises if more than one key differs, so a run that would confound
two changes never starts.

Two of the five set a key the parent does not have at all. A key appearing counts as one
difference in exactly the same way a key changing value does, which is what the check is
written to do. The settings each intervention needs, the weights, the gamma, the priors,
the target count, the thresholds, sit in `observed`, which the check ignores, because they
exist only as part of the one change named in the key and are not separate decisions.

In [7]:
def config_for(run_id, mode):
    spec = INTERVENTIONS[run_id]
    config = dict(PARENT)
    config["run_id"] = run_id
    config["parent"] = PARENT["run_id"]
    config[spec["key"]] = spec["value"]
    config["observed"] = {
        "mode": mode,
        "intervention": run_id,
        "changes": spec["changes"],
        "input": PARENT["observed"]["input"],
        "window": WINDOW,
        "stride": STRIDE,
        "lstm_units": sq.LSTM_UNITS,
        "n_classes": len(CLASSES),
        "classes": CLASSES,
        "features": FEATURES,
        "scaler_fitted_by": PARENT["observed"]["scaler_fitted_by"],
        "model": sq.describe(len(FEATURES), len(CLASSES), window=WINDOW, lstm_units=sq.LSTM_UNITS),
        "compared_against": PARENT["run_id"],
        "notebook": NOTEBOOK,
        "git_sha": GIT_SHA,
        "run_date": RUN_DATE,
    }
    config["observed"]["changed_from_parent"] = sorted(rn.assert_single_change(config, PARENT))
    return config


CONFIGS = {run_id: config_for(run_id, "full") for run_id in INTERVENTIONS}

rows = []
for run_id, config in CONFIGS.items():
    key = INTERVENTIONS[run_id]["key"]
    rows.append(
        {
            "run_id": run_id,
            "the one key": key,
            "parent": PARENT.get(key, "(the parent does not have this key)"),
            "this run": config[key],
            "changed": ", ".join(config["observed"]["changed_from_parent"]),
        }
    )
print(f"parent: {PARENT['run_id']}, read from {PARENT_DIR / 'config.json'}")
print()
print(pd.DataFrame(rows).to_string(index=False))
print()
print(f"ignored by the check, being descriptive rather than experimental: "
      f"{', '.join(rn.IGNORED_KEYS)}")

for run_id, config in CONFIGS.items():
    changed = config["observed"]["changed_from_parent"]
    assert changed == [INTERVENTIONS[run_id]["key"]], (
        f"{run_id} differs from its parent in {changed}, not in the one key it names"
    )

parent: sequence_cnn_lstm_19class, read from /content/repo/data/processed/NB06/config.json

             run_id   the one key                              parent                                                                 this run       changed
class_weighted_loss  class_weight                                None                     inverse frequency, N / (K * n), normalised to mean 1  class_weight
         focal_loss          loss            categorical_crossentropy                    categorical_focal_crossentropy, gamma 2.0, alpha 0.25          loss
   logit_adjustment decision_rule (the parent does not have this key)        logit adjustment, tau 1.0, priors counted on the training windows decision_rule
   threshold_tuning decision_rule (the parent does not have this key) one threshold per class, tuned on validation, largest divided score wins decision_rule
  window_resampling    resampling (the parent does not have this key)     window-level oversampling to the median class siz

Every run's metrics file carries the same three fields beyond the usual scores, and all
three are written by the statement that fits the model.

`caveats` names the classes whose per-class F1 rests on too few windows to read, with
their counts in each partition.

`comparison` is the five classes against the parent, per class, with a detected flag at
0.50 on each side.

`volumetric` is the eight flooding classes against the parent, per class, with the mean
across the eight. That mean is the number that says what an intervention cost the classes
it was not aimed at.

A run and its parent are scored on the same windows of the same partition, so every
difference in those two blocks is a difference on the same items. That is stated in the
blocks themselves. In the fast pass it is not true, because the fast pass scores a sample,
and the line says so there instead.

In [8]:
def items_note(metrics):
    if metrics["n_test"] == PARENT_METRICS["n_test"]:
        return (
            f"both runs are scored on the same {metrics['n_test']:,} test windows of the "
            "two-tier split, so every difference here is a difference on the same items"
        )
    return (
        f"{PARENT_METRICS['n_test']:,} test windows for the parent, {metrics['n_test']:,} "
        "here. This run was scored on a sample of the partition, so the two sides are not "
        "the same items."
    )


def make_extra_metrics():
    def extra(metrics):
        caveats = sq.thin_class_caveats(
            sequences=SEQUENCES, flagged=FLAGGED, thin_below=THIN_BELOW
        )
        caveats["counts_from"] = (
            f"{MANIFEST_PATH.name}, the sequence counts of the whole corpus at window "
            f"{WINDOW} and stride {STRIDE}"
        )
        note = items_note(metrics)
        return {
            "caveats": caveats,
            "comparison": iv.comparison_block(
                metrics, PARENT_METRICS, run_id=PARENT["run_id"], classes=H2_CLASSES,
                threshold=DETECTED_AT, note=note,
            ),
            "volumetric": iv.comparison_block(
                metrics, PARENT_METRICS, run_id=PARENT["run_id"], classes=VOLUMETRIC,
                threshold=DETECTED_AT, note=note, with_mean=True,
            ),
        }

    return extra


preview = pd.DataFrame(
    {
        "class": H2_CLASSES + VOLUMETRIC,
        "group": ["the five"] * 5 + ["volumetric"] * 8,
        "parent_f1": [PARENT_METRICS["per_class_f1"][c] for c in H2_CLASSES + VOLUMETRIC],
        "detected": [PARENT_METRICS["per_class_f1"][c] >= DETECTED_AT for c in H2_CLASSES + VOLUMETRIC],
    }
)
print("what every run below is compared against, class by class")
print(preview.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()
print(f"mean across the eight: {np.mean([PARENT_METRICS['per_class_f1'][c] for c in VOLUMETRIC]):.4f}")

what every run below is compared against, class by class
                  class      group  parent_f1  detected
          Recon-VulScan   the five     0.5385      True
          Recon-OS_Scan   the five     0.6061      True
MQTT-DDoS-Publish_Flood   the five     0.0796     False
               Spoofing   the five     0.7653      True
    MQTT-Malformed_Data   the five     0.8421      True
              DDoS-ICMP volumetric     0.7721      True
               DDoS-SYN volumetric     0.8926      True
               DDoS-TCP volumetric     0.8137      True
               DDoS-UDP volumetric     0.9881      True
               DoS-ICMP volumetric     0.3178     False
                DoS-SYN volumetric     0.8023      True
                DoS-TCP volumetric     0.5161      True
                DoS-UDP volumetric     0.9880      True

mean across the eight: 0.7613


The windows are read from the arrays the preprocessing step wrote, already scaled by a
scaler fitted on the training partition and nothing else. No CSV is opened here.

The validation windows are loaded only when a run needs them, which is the threshold run
and no other, and released as soon as it is finished with them. The fast pass takes a stratified sample of each partition, so every class
keeps its share and the classes with a handful of windows keep at least one.

In [9]:
FAST_CAPS = {"train": 6_000, "val": 3_000, "test": 3_000}
RESUME = True
PREDICT_BATCH = 512
FIT_VERBOSE = 2


def section(title):
    print()
    print("-" * 100)
    print(title)
    print("-" * 100)


def banner(lines):
    print()
    print("#" * 100)
    for line in lines:
        print(f"#  {line[:95]:<96}#")
    print("#" * 100)


def read_partition(array_dir, name, fast, fell_back):
    path = array_dir / f"sequences_{name}.npz"
    if not path.exists():
        raise FileNotFoundError(f"{path} is missing. The preprocessing step writes it.")
    with np.load(path, allow_pickle=False) as npz:
        if [str(v) for v in npz["features"]] != FEATURES:
            raise ValueError(f"{path.name} holds different columns than the manifest lists")
        if [str(v) for v in npz["classes"]] != CLASSES:
            raise ValueError(f"{path.name} holds different classes than the manifest lists")
        if (int(npz["window"]), int(npz["stride"])) != (WINDOW, STRIDE):
            raise ValueError(f"{path.name} was cut at a different window or stride")
        X, y = npz["X"], npz["y"].astype("int64")
    if fast and fell_back:
        index = rn.stratified_subsample(y, cap=FAST_CAPS[name], seed=SEED)
        X, y = X[index], y[index]
        gc.collect()
    print(f"  {path.name:<24} {str(X.shape):>24}   {X.dtype}")
    assert X.shape[1:] == (WINDOW, len(FEATURES))
    assert np.isfinite(X).all(), f"{path.name} holds a value that is not finite"
    return X, y


def array_source(fast: bool):
    """Where the arrays come from, and whether the fast pass has to sample them."""
    mode = "fast" if fast else "full"
    array_dir = NB04_DIRS[mode]
    fell_back = not array_dir.exists()
    if fell_back:
        array_dir = NB04_DIRS["full"]
        if fast:
            print(f"{NB04_DIRS['fast']} is not on Drive, so the fast pass samples the full")
            print("arrays instead. Same windows, fewer of them, same class proportions.")
    return array_dir, fell_back


def load_validation(fast: bool) -> dict:
    """The validation windows, read only when a run needs them."""
    section("Loading the validation windows")
    array_dir, fell_back = array_source(fast)
    X, y = read_partition(array_dir, "val", fast, fell_back)
    print(f"  {len(y):,} validation windows, read for choosing thresholds and nothing else")
    return {"X": sq.reshape(X), "y": y}


def load_sequences(fast: bool) -> dict:
    section("Loading the training and test windows")
    array_dir, fell_back = array_source(fast)

    data = {"array_dir": str(array_dir), "fast": fast, "fell_back": fell_back}
    for name in ("train", "test"):
        X, y = read_partition(array_dir, name, fast, fell_back)
        data[f"X_{name}"], data[f"y_{name}"] = sq.reshape(X), y

    counts = pd.DataFrame(
        {
            "class": CLASSES,
            "train": np.bincount(data["y_train"], minlength=len(CLASSES)),
            "test": np.bincount(data["y_test"], minlength=len(CLASSES)),
        }
    ).sort_values("train")
    print()
    print(f"read from {array_dir}")
    print(f"train {len(data['y_train']):,} windows, test {len(data['y_test']):,}")
    print()
    print(counts.to_string(index=False))

    assert counts[["train", "test"]].min().min() > 0, "a class is missing from one side"
    if not fast:
        for name in ("train", "test"):
            got = {c: int(n) for c, n in zip(CLASSES, np.bincount(data[f"y_{name}"],
                                                                 minlength=len(CLASSES)))}
            assert got == SEQUENCES[name], (
                f"the {name} array holds different per-class counts than the manifest records"
            )
    return data

The training step is one statement.

It checks the configuration against the parent, sets the seed, builds the model, fits it,
turns the fitted model's scores into answers by whichever rule this run uses, scores, and
writes five files: the configuration, the metrics, the true labels, the predicted labels
and the model. A run that does not finish writes nothing, which is the point, because a
score that reached the screen and not the disk cannot be checked afterwards.

The labels are int8 codes and the list they index is in the metrics file. A checkpoint is
written to Drive after every epoch.

In [10]:
def train(run_id, config, data, out_dir):
    """Prepare the one intervention, then fit and save in one statement."""
    hooks, observed = prepare(run_id, data)
    config["observed"].update(observed)
    config["observed"]["n_train"] = int(len(hooks["y_train"]))
    config["observed"]["n_test"] = int(len(data["y_test"]))
    config["observed"]["sequences_from"] = data["array_dir"]
    return sq.fit_and_save(
        out_dir,
        run_id,
        X_train=hooks["X_train"],
        y_train=hooks["y_train"],
        X_test=data["X_test"],
        y_test=data["y_test"],
        classes=CLASSES,
        config=config,
        parent=PARENT,
        window=WINDOW,
        n_features=len(FEATURES),
        lstm_units=sq.LSTM_UNITS,
        seed=SEED,
        extra_metrics=make_extra_metrics(),
        checkpoint=True,
        predict_batch_size=PREDICT_BATCH,
        verbose=FIT_VERBOSE,
        loss=hooks["loss"],
        class_weight=hooks["class_weight"],
        decision_rule=hooks["decision_rule"],
    )


def report(run):
    """The headline numbers of one run, then the two comparison blocks."""
    metrics, config = run["metrics"], run["config"]
    five, eight = metrics["comparison"], metrics["volumetric"]
    print(f"{config['run_id']}   one change from {config['parent']}: "
          f"{', '.join(config['observed']['changed_from_parent'])}")
    print(f"  trained on {metrics['n_train']:,} windows, tested on {metrics['n_test']:,}")
    print(f"  accuracy {metrics['accuracy']:.4f}, weighted F1 {metrics['weighted_f1']:.4f}, "
          f"macro F1 {metrics['macro_f1']:.4f} ({five['macro_f1']['difference']:+.4f})")
    print(f"  of the five, detected at {DETECTED_AT:.2f}: {five['n_detected_this_run']} "
          f"against {five['n_detected_published']} for the parent")
    print(f"  mean F1 across the eight: {eight['mean_f1']['this_run']:.4f} "
          f"({eight['mean_f1']['difference']:+.4f})")
    print(f"  train {metrics['train_seconds']:,.1f}s, predict "
          f"{metrics['inference_seconds']:,.1f}s")
    print(f"  saved to {run['run_dir']}")
    print()
    for title, block, classes in (("the five", five, H2_CLASSES),
                                  ("the eight", eight, VOLUMETRIC)):
        table = pd.DataFrame(
            [
                {
                    "class": label,
                    "parent": block["per_class_f1"][label]["published"],
                    "this run": block["per_class_f1"][label]["this_run"],
                    "difference": block["per_class_f1"][label]["difference"],
                    "detected": block["per_class_f1"][label]["detected_this_run"],
                }
                for label in classes
            ]
        ).sort_values("difference")
        print(f"  {title}")
        print(textwrap.indent(
            table.to_string(index=False, float_format=lambda v: f"{v:.4f}"), "  "))
        print()

Before the full pass starts, a gate on each run.

A run has to be a run before it is worth reading: five files on disk, predictions stored
as codes that index the label list, all nineteen classes scored, finite scores, a caveats
field that flags the thin classes and leaves none out, both comparison blocks present with
every difference recomputed from the per-class scores rather than taken on trust, a
configuration one key from the parent and that key the one the intervention names, and a
saved model that loads again with the published encoder inside it.

The recomputation is the check that matters most here. A comparison block is a subtraction
between two files, and a subtraction written into a metrics file by the run that benefits
from it should be checked against the two numbers it came from.

In [11]:
def gate(run, *, mode) -> pd.DataFrame:
    """Every check a run has to pass. Raises if any of them fails."""
    run_dir = Path(run["run_dir"])
    metrics, config = run["metrics"], run["config"]
    run_id = config["run_id"]
    checks = []

    def check(name, ok, detail):
        checks.append({"check": name, "result": "PASS" if ok else "FAIL", "detail": detail})

    files = sorted(p.name for p in run_dir.iterdir())
    wanted = ["config.json", "metrics.json", "model.keras", "y_pred.npy", "y_true.npy"]
    check("the five files are on disk", all(f in files for f in wanted), ", ".join(files))

    y_true, y_pred = np.load(run_dir / "y_true.npy"), np.load(run_dir / "y_pred.npy")
    check(
        "labels are int8 codes indexing the label list",
        y_true.dtype == np.int8 and y_pred.dtype == np.int8 and metrics["labels"] == CLASSES
        and int(y_true.min()) >= 0 and int(max(y_true.max(), y_pred.max())) < len(CLASSES),
        f"{y_true.dtype}, {len(metrics['labels'])} labels, codes {int(y_true.min())} to "
        f"{int(max(y_true.max(), y_pred.max()))}",
    )
    check(
        "all 19 classes were scored",
        len(set(y_true.tolist())) == len(CLASSES) and len(metrics["per_class_f1"]) == len(CLASSES),
        f"{len(set(y_true.tolist()))} classes present, {len(metrics['per_class_f1'])} scores",
    )
    scores = [metrics["accuracy"], metrics["weighted_f1"], metrics["macro_f1"]]
    losses = metrics.get("history", {}).get("loss", [])
    check(
        "scores and loss are finite",
        all(np.isfinite(v) and 0.0 <= v <= 1.0 for v in scores)
        and bool(losses) and all(np.isfinite(v) for v in losses),
        f"macro {metrics['macro_f1']:.4f}, {len(losses)} epochs, last loss "
        f"{losses[-1]:.4f}" if losses else "no history",
    )

    caveats = metrics["caveats"]
    check(
        "the caveats field flags the thin classes and leaves none out",
        sorted(e["label"] for e in caveats["flagged"]) == sorted(FLAGGED)
        and not caveats["unflagged_and_thin"],
        f"flagged {', '.join(sorted(e['label'] for e in caveats['flagged']))}",
    )

    for field, classes, with_mean in (("comparison", H2_CLASSES, False),
                                      ("volumetric", VOLUMETRIC, True)):
        block = metrics[field]
        recomputed = all(
            abs(block["per_class_f1"][c]["this_run"] - metrics["per_class_f1"][c]) < 1e-12
            and abs(block["per_class_f1"][c]["published"] - PARENT_METRICS["per_class_f1"][c]) < 1e-12
            and abs(block["per_class_f1"][c]["difference"]
                    - (metrics["per_class_f1"][c] - PARENT_METRICS["per_class_f1"][c])) < 1e-12
            for c in classes
        )
        if with_mean:
            mean_here = float(np.mean([metrics["per_class_f1"][c] for c in classes]))
            recomputed = recomputed and abs(block["mean_f1"]["this_run"] - mean_here) < 1e-12
        check(
            f"the {field} block covers its classes and its differences recompute",
            sorted(block["per_class_f1"]) == sorted(classes)
            and block["against"] == PARENT["run_id"] and recomputed,
            f"{len(block['per_class_f1'])} classes against {block['against']}"
            + (f", mean {block['mean_f1']['this_run']:.4f}" if with_mean else ""),
        )

    changed = sorted(rn.assert_single_change(config, PARENT))
    check(
        "one key from the parent, and it is the key this intervention names",
        changed == [INTERVENTIONS[run_id]["key"]],
        f"{config['parent']} to {run_id}, changed {', '.join(changed) if changed else 'nothing'}",
    )

    saved = keras.saving.load_model(run_dir / "model.keras")
    encoder = sq.encoder_matches_baseline(saved, len(FEATURES), len(CLASSES))
    check(
        "the saved model loads and still holds the published encoder",
        encoder["agrees"] and saved.input_shape == (None, WINDOW, len(FEATURES), 1),
        f"{saved.count_params():,} parameters, encoder {encoder['encoder_params']:,}",
    )
    del saved
    gc.collect()

    table = pd.DataFrame(checks)
    print(f"  gate on {run_id}, {mode} pass")
    print(textwrap.indent(table.to_string(index=False), "  "))
    failed = table[table["result"] == "FAIL"]
    if len(failed):
        raise AssertionError(
            f"{run_id} failed {len(failed)} check(s) on the {mode} pass: "
            f"{', '.join(failed['check'].tolist())}. Nothing further runs."
        )
    print(f"  {len(table)} of {len(table)} passed")
    return table

The order the five run in is a decision.

A full pass is five trainings and a session can end inside one, so the question is which
results I would rather have if it stops early. The two that change the loss go first,
because they are the ones that change what the model learns and they are the cheapest to
reason about afterwards. The two that only change how a trained model decides go next.
The resampling run goes last, because it trains on more windows than the others and takes
the longest.

A finished run is read back off disk rather than fitted again, so a session that dies
after three picks up at the fourth.

In [12]:
RUN_ORDER = [
    "class_weighted_loss",
    "focal_loss",
    "logit_adjustment",
    "threshold_tuning",
    "window_resampling",
]

assert sorted(RUN_ORDER) == sorted(INTERVENTIONS), "the order and the interventions disagree"


def run_pass(fast: bool) -> dict:
    mode = "fast" if fast else "full"
    out_dir = OUT_DIRS[mode]
    out_dir.mkdir(parents=True, exist_ok=True)

    started = time.time()
    result = {"mode": mode, "out_dir": out_dir, "runs": {}, "checks": {}, "read_back": []}
    data = None

    for position, run_id in enumerate(RUN_ORDER, start=1):
        section(f"Run {position} of {len(RUN_ORDER)} - {run_id} - {mode} pass")
        existing = sq.load_run(out_dir, run_id) if RESUME else None

        if existing is not None:
            print(f"{run_id}: already complete, read back from {existing['run_dir']}, "
                  f"macro F1 {existing['metrics']['macro_f1']:.4f}")
            run = existing
            result["read_back"].append(run_id)
        else:
            if data is None:
                data = load_sequences(fast)
            if INTERVENTIONS[run_id]["needs_validation"] and "validation" not in data:
                data["validation"] = load_validation(fast)
            run = train(run_id, config_for(run_id, mode), data, out_dir)
            print()

        report(run)
        result["runs"][run_id] = run
        result["checks"][run_id] = gate(run, mode=mode)

        if data is not None and "validation" in data and not any(
            INTERVENTIONS[r]["needs_validation"] for r in RUN_ORDER[position:]
        ):
            data.pop("validation")
            gc.collect()
            print("  the validation windows are released; no run left needs them")

    data = None
    gc.collect()
    result["elapsed_s"] = time.time() - started
    print()
    print(f"{len(result['runs'])} runs in {out_dir}, "
          f"{len(result['read_back'])} read back, {result['elapsed_s'] / 60:.1f} minutes")
    return result

Before starting it, how long it will take.

The rate is not guessed this time. The parent run recorded how many windows it trained on
and how many seconds that took, so the number of gradient steps a second is arithmetic on
its metrics file. The step counts below are exact, and the only assumption is that these
five runs go at the parent's rate, which the resampling run will not quite do because it
trains on more windows.

In [13]:
PARENT_STEPS = int(np.ceil(PARENT_METRICS["n_train"] / BATCH_SIZE)) * EPOCHS
STEP_RATE = PARENT_STEPS / PARENT_METRICS["train_seconds"]

train_windows = MANIFEST["arrays"]["sequences_train"]["shape"][0]
median_target = int(np.median([n for n in SEQUENCES["train"].values() if n > 0]))
resampled = train_windows + sum(
    max(0, median_target - n) for n in SEQUENCES["train"].values() if n > 0
)

rows = []
for mode in ("fast", "full"):
    for run_id in RUN_ORDER:
        n_train = FAST_CAPS["train"] if mode == "fast" else train_windows
        if run_id == "window_resampling":
            n_train = int(round(n_train * resampled / train_windows))
        steps = int(np.ceil(n_train / BATCH_SIZE)) * EPOCHS
        done = RESUME and rn.load_run(OUT_DIRS[mode], run_id) is not None
        rows.append(
            {
                "pass": mode,
                "run_id": run_id,
                "train_windows": n_train,
                "gradient_steps": steps,
                "minutes": 0.0 if done else steps / STEP_RATE / 60,
                "status": "already complete" if done else "to run",
            }
        )
ESTIMATE = pd.DataFrame(rows)

print(f"the parent trained on {PARENT_METRICS['n_train']:,} windows in "
      f"{PARENT_METRICS['train_seconds']:,.0f}s, which is {PARENT_STEPS:,} gradient steps "
      f"at {STEP_RATE:,.1f} a second")
print(f"oversampling to the median class size, {median_target:,}, takes the training set "
      f"from {train_windows:,} windows to {resampled:,}")
print()
print(ESTIMATE.to_string(index=False, float_format=lambda v: f"{v:,.1f}"))
print()
print(f"fast pass {ESTIMATE.loc[ESTIMATE['pass'] == 'fast', 'minutes'].sum():,.1f} minutes, "
      f"full pass {ESTIMATE.loc[ESTIMATE['pass'] == 'full', 'minutes'].sum():,.1f} minutes "
      f"({ESTIMATE.loc[ESTIMATE['pass'] == 'full', 'minutes'].sum() / 60:,.1f} hours)")
print()
print("Loading the arrays and scoring are not in those numbers, and neither is the")
print("threshold search. If the full pass is longer than the session available, stop here:")
print("finished runs are read back rather than refitted, so it can be done across two.")

the parent trained on 249,061 windows in 2,428s, which is 77,840 gradient steps at 32.1 a second
oversampling to the median class size, 6,017, takes the training set from 249,061 windows to 295,925

pass              run_id  train_windows  gradient_steps  minutes status
fast class_weighted_loss           6000            1880      1.0 to run
fast          focal_loss           6000            1880      1.0 to run
fast    logit_adjustment           6000            1880      1.0 to run
fast    threshold_tuning           6000            1880      1.0 to run
fast   window_resampling           7129            2230      1.2 to run
full class_weighted_loss         249061           77840     40.5 to run
full          focal_loss         249061           77840     40.5 to run
full    logit_adjustment         249061           77840     40.5 to run
full    threshold_tuning         249061           77840     40.5 to run
full   window_resampling         295925           92480     48.1 to run

fast pas

Both passes run here, fast first. Every run in the fast pass goes through the same gate
the full pass uses, and a failure anywhere stops the cell before the long pass starts.

In [14]:
banner([
    "FAST PASS",
    "all five interventions, every check, on a sample of the windows",
    "for shapes and plumbing only",
    "not a result, and never entered in the ledger",
])
FAST = run_pass(True)

banner([
    "FULL PASS",
    f"all five interventions on every window, {EPOCHS} epochs at batch {BATCH_SIZE}",
    "this is the pass that goes in the ledger",
])
FULL = run_pass(False)

banner([
    f"fast pass {FAST['elapsed_s'] / 60:.1f} min, full pass {FULL['elapsed_s'] / 60:.1f} min",
    f"{len(FULL['runs'])} runs in {FULL['out_dir']}",
    f"{len(FULL['read_back'])} were read back rather than refitted",
])


####################################################################################################
#  FAST PASS                                                                                       #
#  all five interventions, every check, on a sample of the windows                                 #
#  for shapes and plumbing only                                                                    #
#  not a result, and never entered in the ledger                                                   #
####################################################################################################

----------------------------------------------------------------------------------------------------
Run 1 of 5 - class_weighted_loss - fast pass
----------------------------------------------------------------------------------------------------

----------------------------------------------------------------------------------------------------
Loading the training and test windows
-----


threshold_tuning   one change from sequence_cnn_lstm_19class: decision_rule
  trained on 181 windows, tested on 80
  accuracy 0.5750, weighted F1 0.4901, macro F1 0.4808 (-0.2330)
  of the five, detected at 0.50: 3 against 4 for the parent
  mean F1 across the eight: 0.3150 (-0.4463)
  train 13.9s, predict 2.7s
  saved to /content/drive/MyDrive/AG_PRAXIS_artifacts/NB07_fast/threshold_tuning

  the five
                    class  parent  this run  difference  detected
            Recon-VulScan  0.5385    0.0000     -0.5385     False
      MQTT-Malformed_Data  0.8421    0.6667     -0.1754      True
                 Spoofing  0.7653    0.6667     -0.0986      True
  MQTT-DDoS-Publish_Flood  0.0796    0.0000     -0.0796     False
            Recon-OS_Scan  0.6061    0.6667      0.0606      True

  the eight
      class  parent  this run  difference  detected
    DoS-UDP  0.9880    0.0000     -0.9880     False
    DoS-SYN  0.8023    0.0000     -0.8023     False
    DoS-TCP  0.5161    0.000


window_resampling   one change from sequence_cnn_lstm_19class: resampling
  trained on 181 windows, tested on 80
  accuracy 0.4750, weighted F1 0.4283, macro F1 0.4158 (-0.2980)
  of the five, detected at 0.50: 2 against 4 for the parent
  mean F1 across the eight: 0.2867 (-0.4746)
  train 13.9s, predict 2.5s
  saved to /content/drive/MyDrive/AG_PRAXIS_artifacts/NB07_fast/window_resampling

  the five
                    class  parent  this run  difference  detected
      MQTT-Malformed_Data  0.8421    0.2500     -0.5921     False
            Recon-VulScan  0.5385    0.0000     -0.5385     False
                 Spoofing  0.7653    0.6667     -0.0986      True
  MQTT-DDoS-Publish_Flood  0.0796    0.0000     -0.0796     False
            Recon-OS_Scan  0.6061    0.6667      0.0606      True

  the eight
      class  parent  this run  difference  detected
    DoS-UDP  0.9880    0.0000     -0.9880     False
   DDoS-SYN  0.8926    0.0000     -0.8926     False
    DoS-SYN  0.8023    0.0000

Every run's full per-class table, one after another, sorted so the classes each
intervention did least for are at the top.

In [15]:
for run_id in RUN_ORDER:
    metrics = FULL["runs"][run_id]["metrics"]
    table = rn.per_class_frame(metrics)
    table["parent"] = [PARENT_METRICS["per_class_f1"][c] for c in table["label"]]
    table["difference"] = table["f1"] - table["parent"]
    table["thin"] = table["label"].isin(FLAGGED)
    table = table[["label", "parent", "f1", "difference", "support", "thin"]].sort_values(
        "difference"
    )
    print("=" * 100)
    print(f"{run_id}   macro F1 {metrics['macro_f1']:.4f}, weighted F1 "
          f"{metrics['weighted_f1']:.4f}, accuracy {metrics['accuracy']:.4f}")
    print("=" * 100)
    print(table.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    gained = int((table["difference"] > 0).sum())
    lost = int((table["difference"] < 0).sum())
    print(f"gained {gained}, lost {lost}, unchanged {len(table) - gained - lost}; "
          f"sum of gains {table.loc[table['difference'] > 0, 'difference'].sum():+.4f}, "
          f"sum of losses {table.loc[table['difference'] < 0, 'difference'].sum():+.4f}")
    print()

class_weighted_loss   macro F1 0.6481, weighted F1 0.6965, accuracy 0.7056
                  label  parent     f1  difference  support  thin
               DDoS-TCP  0.8137 0.3996     -0.4142     7302 False
               Spoofing  0.7653 0.4295     -0.3358      104 False
    MQTT-Malformed_Data  0.8421 0.6000     -0.2421       40  True
               DDoS-SYN  0.8926 0.7166     -0.1761     6894 False
          Recon-VulScan  0.5385 0.3729     -0.1656       18  True
                DoS-SYN  0.8023 0.6959     -0.1064     3942 False
                 Benign  0.9808 0.8966     -0.0842     1381 False
                DoS-TCP  0.5161 0.4643     -0.0519     3282 False
                DoS-UDP  0.9880 0.9509     -0.0371     5501 False
               DDoS-UDP  0.9881 0.9531     -0.0349     6255 False
 MQTT-DoS-Connect_Flood  0.9741 0.9394     -0.0347       94 False
              DDoS-ICMP  0.7721 0.7589     -0.0132     7826 False
MQTT-DDoS-Connect_Flood  0.9996 0.9885     -0.0111     1288 False
 

One table for all of it.

Each row is a run. The first row is the parent, which ran no intervention and is here as
the line the others are read against. The next five columns are the per-class F1 of the
five classes the comparison is about. Then the mean F1 across the eight flooding classes,
then macro-F1 over all nineteen. Gains and costs on the same line.

In [16]:
def summary_row(run_id, metrics):
    row = {"run": run_id}
    for label in H2_CLASSES:
        row[label] = metrics["per_class_f1"][label]
    row["volumetric_mean_f1"] = float(np.mean([metrics["per_class_f1"][c] for c in VOLUMETRIC]))
    row["macro_f1"] = metrics["macro_f1"]
    return row


SUMMARY = pd.DataFrame(
    [summary_row(f"{PARENT['run_id']} (parent)", PARENT_METRICS)]
    + [summary_row(run_id, FULL["runs"][run_id]["metrics"]) for run_id in RUN_ORDER]
)
for column in SUMMARY.columns:
    if column != "run" and not pd.api.types.is_numeric_dtype(SUMMARY[column]):
        raise TypeError(f"{column} came out as {SUMMARY[column].dtype}, not numeric")

print("=" * 130)
print("NB07 full pass - every intervention, the five classes, the eight, and macro F1")
print("=" * 130)
print(SUMMARY.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()
print("Columns: the run, then the F1 of each of the five classes the comparison is about,")
print("then the mean F1 across the eight flooding classes, then macro F1 over all nineteen.")
print(f"The parent row ran no intervention. A class counts as detected at F1 {DETECTED_AT:.2f}.")

NB07 full pass - every intervention, the five classes, the eight, and macro F1
                               run  Recon-VulScan  Recon-OS_Scan  MQTT-DDoS-Publish_Flood  Spoofing  MQTT-Malformed_Data  volumetric_mean_f1  macro_f1
sequence_cnn_lstm_19class (parent)         0.5385         0.6061                   0.0796    0.7653               0.8421              0.7613    0.7138
               class_weighted_loss         0.3729         0.6667                   0.0717    0.4295               0.6000              0.6614    0.6481
                        focal_loss         0.3333         0.3377                   0.0631    0.6667               0.9500              0.7524    0.6838
                  logit_adjustment         0.4746         0.5730                   0.0631    0.7442               0.8706              0.7720    0.7238
                  threshold_tuning         0.3902         0.8193                   0.0804    0.8406               0.8732              0.7643    0.7320
               

One ledger entry per run, and one block covering the five together, ready to paste into
`RESULTS_LEDGER.md`. The single-change check runs once more on each configuration as it
was written to disk.

In [17]:
STATUS = "reference run" + (", working tree dirty" if GIT_DIRTY else "")

for run_id in RUN_ORDER:
    saved = json.loads((Path(FULL["runs"][run_id]["run_dir"]) / "config.json").read_text())
    changed = sorted(rn.assert_single_change(saved, PARENT))
    assert changed == [INTERVENTIONS[run_id]["key"]], f"{run_id} is not one key from its parent"
print(f"all {len(RUN_ORDER)} configurations on disk are one key from {PARENT['run_id']}")
print()

overall = f"""
### NB07 — class balancing ({RUN_DATE})

| field | value |
|---|---|
| notebook | {NOTEBOOK} |
| run date | {RUN_DATE} |
| git sha | {GIT_SHA}{" (working tree dirty)" if GIT_DIRTY else ""} |
| seed | {SEED} |
| pass reported | full |
| runs | {len(RUN_ORDER)}, one intervention each, of which {len(FULL["read_back"])} were read back from an earlier session |
| runtime | fast {FAST["elapsed_s"] / 60:.1f} min, full {FULL["elapsed_s"] / 60:.1f} min |
| parent | {PARENT["run_id"]}, macro F1 {PARENT_METRICS["macro_f1"]:.4f} |
| input | {WINDOW} records at stride {STRIDE}, {len(FEATURES)} features, two-tier split |
| role | reported result, per-class benefit and cost. Not a hypothesis test, no significance computed |
| artifacts | {FULL["out_dir"]} |
| status | {STATUS} |

| run | {" | ".join(H2_CLASSES)} | volumetric mean | macro F1 |
|---|{"---|" * (len(H2_CLASSES) + 2)}
""" + "\n".join(
    "| " + row["run"] + " | "
    + " | ".join(f"{row[label]:.4f}" for label in H2_CLASSES)
    + f" | {row['volumetric_mean_f1']:.4f} | {row['macro_f1']:.4f} |"
    for _, row in SUMMARY.iterrows()
)

blocks = [overall]
for position, run_id in enumerate(RUN_ORDER, start=1):
    run = FULL["runs"][run_id]
    metrics, config = run["metrics"], run["config"]
    five, eight = metrics["comparison"], metrics["volumetric"]
    weakest = sorted(metrics["per_class_f1"].items(), key=lambda pair: pair[1])[:4]
    blocks.append(f"""
### NB07 — {run_id} ({RUN_DATE})

| field | value |
|---|---|
| notebook | {NOTEBOOK} |
| git sha | {GIT_SHA}{" (working tree dirty)" if GIT_DIRTY else ""} |
| seed | {config["seed"]} |
| written | {position} of {len(RUN_ORDER)} |
| intervention | {config["observed"]["intervention"]}, changes {config["observed"]["changes"]} |
| parent | {config["parent"]} |
| the one change | {", ".join(config["observed"]["changed_from_parent"])} = {config[INTERVENTIONS[run_id]["key"]]} |
| training windows | {metrics["n_train"]:,} |
| test windows | {metrics["n_test"]:,} |
| accuracy | {metrics["accuracy"]:.4f} |
| weighted P / R / F1 | {metrics["weighted_precision"]:.4f} / {metrics["weighted_recall"]:.4f} / {metrics["weighted_f1"]:.4f} |
| macro P / R / F1 | {metrics["macro_precision"]:.4f} / {metrics["macro_recall"]:.4f} / {metrics["macro_f1"]:.4f} |
| macro F1 against the parent | {five["macro_f1"]["this_run"]:.4f} against {five["macro_f1"]["published"]:.4f}, {five["macro_f1"]["difference"]:+.4f} |
| the five classes, F1 | {", ".join(f"{label} {five['per_class_f1'][label]['this_run']:.4f} ({five['per_class_f1'][label]['difference']:+.4f})" for label in H2_CLASSES)} |
| of the five, detected at {DETECTED_AT:.2f} | {five["n_detected_this_run"]} of {five["n_classes_compared"]}, parent {five["n_detected_published"]} of {five["n_classes_compared"]} |
| the eight volumetric classes, mean F1 | {eight["mean_f1"]["this_run"]:.4f} against {eight["mean_f1"]["parent"]:.4f}, {eight["mean_f1"]["difference"]:+.4f} |
| of the eight, detected at {DETECTED_AT:.2f} | {eight["n_detected_this_run"]} of {eight["n_classes_compared"]}, parent {eight["n_detected_published"]} of {eight["n_classes_compared"]} |
| four weakest classes | {", ".join(f"{label} {value:.2f}" for label, value in weakest)} |
| too few windows to interpret | {"; ".join(f"{e['label']} ({e['sequences']['test']} test)" for e in metrics["caveats"]["flagged"])} |
| gate | {len(FULL["checks"][run_id])} checks, all passed |
| train seconds | {metrics["train_seconds"]:,.1f} |
| inference seconds | {metrics["inference_seconds"]:,.1f} |
| artifacts | {run["run_dir"]} |
| status | {STATUS} |
""")

print("=" * 100)
print("paste into RESULTS_LEDGER.md")
print("=" * 100)
print("\n".join(blocks))

all 5 configurations on disk are one key from sequence_cnn_lstm_19class

paste into RESULTS_LEDGER.md

### NB07 — class balancing (2026-08-08)

| field | value |
|---|---|
| notebook | AG_PRAXIS_NB07_class_balancing.ipynb |
| run date | 2026-08-08 |
| git sha | 45f376d |
| seed | 42 |
| pass reported | full |
| runs | 5, one intervention each, of which 0 were read back from an earlier session |
| runtime | fast 1.6 min, full 212.3 min |
| parent | sequence_cnn_lstm_19class, macro F1 0.7138 |
| input | 50 records at stride 25, 44 features, two-tier split |
| role | reported result, per-class benefit and cost. Not a hypothesis test, no significance computed |
| artifacts | /content/drive/MyDrive/AG_PRAXIS_artifacts/NB07 |
| status | reference run |

| run | Recon-VulScan | Recon-OS_Scan | MQTT-DDoS-Publish_Flood | Spoofing | MQTT-Malformed_Data | volumetric mean | macro F1 |
|---|---|---|---|---|---|---|---|
| sequence_cnn_lstm_19class (parent) | 0.5385 | 0.6061 | 0.0796 | 0.7653 | 0.842